# Metrics dashboard — add the Case law pipeline

`metrics_dashboard.html` had **no producer anywhere in the project**: it was built in a
scratchpad in an earlier session and only the artefact survived, so it could not be regenerated
or extended. This notebook is that producer. It does not rebuild the page from nothing — that
would throw away everything already written into it — it **parses the existing HTML and injects**
the Case law pipeline, so every prior section stays byte-identical.

Idempotent: running it twice replaces the injected block rather than adding a second one.

## What gets added

| | |
|---|---|
| nav | a **Case law** entry under *Validation figures* |
| raw data | a row for the Case law source, with the citation below |
| section | `#val-caselaw` — every `case_law_validation_*.jpg`, captioned |
| header stat | the validation-figure count and notebook count |
| CSS | a fifth accent `--p5`, mirroring the `--p1..--p4` rules already in the page |

## The data must be credited

The Case law citation network and metadata were **provided by the authors of**

> Lee, S., Kim, T., Yoon, J., & Youn, H. (2024). *When Common Law Ages: Two Centuries of
> Growing Inertia in US Judicial Opinions.* arXiv preprint arXiv:2410.04493.

That citation is injected in two places — the raw-data table and the head of the Case law
section — so it cannot be read past.

> **2026-09-11 — where new galleries come from.** This notebook only injects the Case law block. Every gallery, including the Dimensions and Cross-checks sections added on 2026-09-11, is now produced by `refresh_dashboard.py` (`FAMILIES` / `CAPTIONS` tables): it discovers every `Figures/<notebook>_<section>.jpg` export, creates the section, nav entry, accent colour, KPI card and raw-data row of a family the page does not have yet, and regenerates the Derived tables inventory via `dashboard_inventory.py`. Adding a validation notebook = one `FAMILIES` entry plus its captions. The final cell here calls that script, so running this notebook end to end is still a complete rebuild.


In [1]:
import os, re, base64, html, datetime
from pathlib import Path

VAL   = Path('/project/jevans/Dawoon/Science of Science/validation')
DASH  = VAL / 'metrics_dashboard.html'
FIGS  = VAL / 'Figures'
BAK   = VAL / f'metrics_dashboard.html.bak-{datetime.date.today():%Y%m%d}'

CITATION_HTML = (
    'Lee, S., Kim, T., Yoon, J., &amp; Youn, H. (2024). '
    '<i>When Common Law Ages: Two Centuries of Growing Inertia in US Judicial Opinions.</i> '
    'arXiv preprint <a href="https://arxiv.org/abs/2410.04493">arXiv:2410.04493</a>.')

# section number -> (title, what the figure tests). Keyed by the suffix val_common.save() wrote.
CAPTIONS = {
    '2':   ('The corpus in time', 'Cases per decision year, cumulative share, and references per case. 1800–2020.'),
    '3':   ('Forward citations &amp; windowed monotonicity', 'C<sub>3</sub> ≤ C<sub>5</sub> ≤ C<sub>10</sub> ≤ C<sub>all</sub> asserted on every case; log–log tail.'),
    '4':   ('Mean citations by decision year', 'Each window drawn faint past the year it stops having elapsed.'),
    '5':   ('Citation age profile', 'When a case is cited, overall and by decision era — the shape the windows cut through.'),
    '6':   ('Disruption — distribution, trend, composition', 'CD spikes at ±1 split by citer count; n<sub>i</sub>/n<sub>j</sub>/n<sub>k</sub> over time.'),
    '7':   ('F / E / G decomposition', 'Foundation / Extension / Generalization shares, asserted to sum to 1.'),
    '8':   ('Hit percentile', 'Percentile within (jurisdiction, year); the block at 0 is the 27% never cited.'),
    '9':   ('Metric means by jurisdiction', 'All 43 jurisdictions above 20,000 cases, citations and disruption.'),
    '10':  ('Cross-metric correlations', 'Spearman on a 500k sample — CD against citations, hit against C<sub>all</sub>.'),
    '11':  ('Sleeping Beauty — B, T and impact', 'B distribution, awakening time, and B rising with T.'),
    '11b': ('Sleeping Beauty across three families', 'Papers, patents and cases on one axis — same kernel, same n_cite floor.'),
    '11c': ('Beauty coefficient survival', 'Share of documents above each B threshold, by family.'),
    '12':  ('Citation trajectory &amp; convexity — top 1%', 'Cumulative 30-year trajectory against the straight line.'),
    '13':  ('Is case law becoming less disruptive?', 'Mean CD by decision year, each window only where it has closed.'),
    '14':  ('Stratified — All / State / Federal × Supreme', 'Disruption, citations and B by court level and bench.'),
    '14b': ('Age profile and F/E/G by stratum', 'Whether the four strata are comparable on fixed windows.'),
}
print(f'dashboard : {DASH}  ({DASH.stat().st_size/1e6:.1f} MB)')
print(f'figures   : {len(list(FIGS.glob("case_law_validation_*.jpg")))} case law jpgs in {FIGS}')

dashboard : /project/jevans/Dawoon/Science of Science/validation/metrics_dashboard.html  (23.7 MB)
figures   : 16 case law jpgs in /project/jevans/Dawoon/Science of Science/validation/Figures


In [2]:
%%time
def fig_card(sec, title, tests, jpg):
    b64 = base64.b64encode(jpg.read_bytes()).decode()
    try:
        from PIL import Image
        w, h = Image.open(jpg).size
    except Exception:
        w = h = None
    dim = f' width="{w}" height="{h}"' if w else ''
    return (f'<figure class="card" data-pipe="caselaw">\n'
            f'      <div class="cardhead">\n'
            f'        <span class="sec p5">§{sec}</span>\n'
            f'        <code class="src">case_law_validation.ipynb</code>\n'
            f'      </div>\n'
            f'      <h3>{title}</h3>\n'
            f'      <p class="tests">{tests}</p>\n'
            f'      <button class="figwrap" type="button" '
            f'aria-label="Enlarge figure: {re.sub("<[^>]+>", "", title)}">'
            f'<img class="figimg" loading="lazy"{dim} alt="case_law_validation_{sec}" '
            f'src="data:image/jpeg;base64,{b64}"></button>\n'
            f'    </figure>')

def sec_key(name):
    s = name.replace('case_law_validation_', '').replace('.jpg', '')
    return (int(re.match(r'\d+', s).group()), s)

jpgs = sorted(FIGS.glob('case_law_validation_*.jpg'), key=lambda p: sec_key(p.name))
cards = []
for p in jpgs:
    sec = p.name.replace('case_law_validation_', '').replace('.jpg', '')
    title, tests = CAPTIONS.get(sec, (f'Section {sec}', ''))
    cards.append(fig_card(sec, title, tests, p))
print(f'{len(cards)} cards built  ({sum(len(c) for c in cards)/1e6:.1f} MB of base64)')

SECTION = (
 '\n<section id="val-caselaw" class="valsec">\n'
 '  <div class="shead sub"><span class="rule p5"></span><h2>Case law</h2>\n'
 '    <span class="chip p5">CASE</span>\n'
 f'    <span class="meta">{len(cards)} figures · case_law_validation.ipynb</span></div>\n'
 '  <p class="intro">5,179,698 US court opinions and 47,519,638 citations between them, 1800–2020.\n'
 '  The same estimators as the paper and patent families — citation windows, the CD index and its\n'
 '  F/E/G decomposition, the beauty coefficient — on a network whose citation culture is different:\n'
 '  precedent is cited immediately and then, occasionally, rediscovered a century later.</p>\n'
 '  <p class="intro" style="border-left:3px solid var(--p5);padding-left:12px">\n'
 '  <b>Data provenance.</b> The citation network and metadata were provided by the authors of '
 f'{CITATION_HTML} Any use of the <code>case_*</code> outputs should cite it.</p>\n'
 '  <div class="gallery">' + '\n'.join(cards) + '</div>\n'
 '</section>\n')
print(f'section built: {len(SECTION)/1e6:.1f} MB')

16 cards built  (16.8 MB of base64)
section built: 16.8 MB
CPU times: user 61.2 ms, sys: 39.8 ms, total: 101 ms
Wall time: 102 ms


In [3]:
%%time
h = DASH.read_text(encoding='utf-8')
if not BAK.exists():
    BAK.write_text(h, encoding='utf-8')
    print(f'backup -> {BAK.name}')

# Idempotent: drop any block this notebook injected before, so a re-run replaces rather than
# appends. Every insertion below is bracketed or uniquely keyed for exactly this reason.
# Replaced with '', not '\\n': SECTION already carries its own leading newline, so putting
# one back here leaks a blank line on every re-run -- the file grew a byte per run.
h = re.sub(r'\n<section id="val-caselaw".*?</section>\n', '', h, flags=re.S)
h = re.sub(r'\s*<li><a href="#val-caselaw">.*?</li>', '', h, flags=re.S)
h = re.sub(r'\s*<tr data-inject="caselaw">.*?</tr>', '', h, flags=re.S)
h = h.replace('  --p5:#e34948;\n', '')
# The CSS rules are delimited, so the cleanup removes exactly what it added. Stripping
# the variable but not the rules is what made the previous run non-idempotent.
h = re.sub(r'\n/\* caselaw-inject \*/.*?/\* /caselaw-inject \*/', '', h, flags=re.S)

# 1) a fifth accent, mirroring the --p1..--p4 rules already in the page
o = '  --p1:#2a78d6; --p2:#eb6834; --p3:#1baf7a; --p4:#4a3aa7;'
assert h.count(o) >= 1
h = h.replace(o, o + '\n  --p5:#e34948;', 1)
o = '.cardhead .sec.p4 { color:var(--p4); }'
assert h.count(o) == 1
h = h.replace(o, o + '\n/* caselaw-inject */'
                    '\n.dot.p5 { background:var(--p5); }'
                    '\n.chip.p5 { color:var(--p5); background:color-mix(in srgb, var(--p5) 12%, transparent); }'
                    '\n.shead .rule.p5 { background:var(--p5); }'
                    '\n.cardhead .sec.p5 { color:var(--p5); }'
                    '\n/* /caselaw-inject */')

# 2) nav entry, after the last validation sub-entry
o = '<li><a href="#val-ppp">'
i = h.find(o)
assert i > 0
j = h.find('</li>', i) + len('</li>')
h = h[:j] + '\n        <li><a href="#val-caselaw"><span class="dot p5"></span>Case law</a></li>' + h[j:]

# 3) the Case law source row in the raw-data table, carrying the citation
row = (
 '\n<tr data-inject="caselaw">\n'
 '      <td><span class="dot p5"></span><b>Case law citation network</b><br>'
 '<code class="path">Science of Science/Case law</code></td>\n'
 '      <td class="num nowrap">5,179,698 opinions · 47,519,638 citation edges</td>\n'
 '      <td><span class="flag info">provided</span></td>\n'
 f'      <td class="note"><code>Edge_list.parquet</code> and <code>metadata.csv</code> — US court '
 f'opinions 1666–2020 with jurisdiction, court, reporter and decision date. '
 f'<b>Provided by the authors of</b> {CITATION_HTML}</td></tr>')
o = '<tbody><tr>\n      <td><span class="dot p1"></span><b>OpenAlex snapshot 2026-01-16</b>'
assert h.count(o) == 1
h = h.replace(o, '<tbody>' + row + '<tr>\n      <td><span class="dot p1"></span>'
                 '<b>OpenAlex snapshot 2026-01-16</b>', 1)

# 4) the section itself, immediately before Open gaps
o = '\n<section id="gaps">'
assert h.count(o) == 1
h = h.replace(o, SECTION + o, 1)

# 5) the header stat: 49 figures / 4 notebooks -> +this pipeline
# Rewritten from whatever it currently says, so a re-run converges instead of skipping.
m = re.search(r'<div class="v">(\d+)</div><div class="k">validation figures</div>'
              r'<div class="sub">(\d+) notebooks</div>', h)
assert m, 'header stat block not found'
base_figs = int(m.group(1)) - (len(cards) if int(m.group(2)) == 5 else 0)
n = (f'<div class="v">{base_figs + len(cards)}</div><div class="k">validation figures</div>'
     f'<div class="sub">5 notebooks</div>')
h = h[:m.start()] + n + h[m.end():]
print(f'header stat: {m.group(1)}/{m.group(2)} -> {base_figs + len(cards)}/5')

DASH.write_text(h, encoding='utf-8')
print(f'WROTE {DASH}  ({DASH.stat().st_size/1e6:.1f} MB)')

header stat: 65/5 -> 65/5
WROTE /project/jevans/Dawoon/Science of Science/validation/metrics_dashboard.html  (23.7 MB)
CPU times: user 615 ms, sys: 70.4 ms, total: 685 ms
Wall time: 688 ms


In [4]:
%%time
# ---- 6) header KPIs, the pipeline count, and the Derived tables section --------------------
import pyarrow.parquet as pq
CL_OUT = Path('/project/jevans/Dawoon/Science of Science/Case law/output')
tabs = sorted(CL_OUT.glob('*.parquet'))
cl_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in tabs)
cl_bytes = sum(f.stat().st_size for f in tabs)
print(f'case law: {len(tabs)} tables, {cl_rows:,} rows, {cl_bytes/1e6:.0f} MB')

h = DASH.read_text(encoding='utf-8')
h = re.sub(r'\s*<div class="kpi" data-inject="caselaw">.*?</div></div>', '', h, flags=re.S)
h = re.sub(r'\s*<tr data-pipe="caselaw">.*?</tr>', '', h, flags=re.S)

# a KPI card for the corpus, placed before the validation-figure count
kpi = ('\n  <div class="kpi" data-inject="caselaw"><div class="v">5.18 M</div>'
       '<div class="k">court opinions</div>'
       '<div class="sub">47.5 M citations · 1800–2020</div></div>')
anchor = '\n  <div class="kpi"><div class="v">65</div><div class="k">validation figures</div>'
if anchor not in h:
    anchor = re.search(r'\n  <div class="kpi"><div class="v">\d+</div><div class="k">validation figures</div>',
                       h).group(0)
h = h.replace(anchor, kpi + anchor, 1)

# the page still said four pipelines and counted only their tables
h = h.replace(
  'Four pipelines turn two bibliographic snapshots and two linkage files into 2.1 billion rows\n  of per-document metrics.',
  'Five pipelines turn two bibliographic snapshots, two linkage files and a court-opinion\n'
  '  citation network into 2.2 billion rows of per-document metrics.')
h = h.replace('<div class="pipebar"><i></i><i></i><i></i><i></i></div>',
              '<div class="pipebar"><i></i><i></i><i></i><i></i><i></i></div>')
m = re.search(r'<div class="v">([\d.]+) B</div><div class="k">metric rows</div>'
              r'<div class="sub">across (\d+) tables</div>', h)
if m:
    base_tabs = int(m.group(2)) - (len(tabs) if int(m.group(2)) > 21 else 0)
    h = (h[:m.start()] + f'<div class="v">{(2.13e9 + cl_rows)/1e9:.2f} B</div>'
         f'<div class="k">metric rows</div>'
         f'<div class="sub">across {base_tabs + len(tabs)} tables</div>' + h[m.end():])
    print(f'metric rows: {m.group(1)} B / {m.group(2)} tables -> '
          f'{(2.13e9 + cl_rows)/1e9:.2f} B / {base_tabs + len(tabs)} tables')

# the Derived tables heading counts files; and the seven case law tables are appended to it
h = re.sub(r'(<h2>Derived tables</h2><span class="meta">)\d+( parquet files</span>)',
           lambda mm: f'{mm.group(1)}{21 + len(tabs)}{mm.group(2)}', h)

def cols_of(f):
    names = pq.ParquetFile(f).schema_arrow.names
    head = ', '.join(names[:6])
    return head + (f' … +{len(names)-6} more' if len(names) > 6 else '')

rows_html = ''.join(
    f'\n<tr data-pipe="caselaw">\n'
    f'      <td><span class="dot p5"></span><code>{f.name}</code></td>\n'
    f'      <td class="num">{pq.ParquetFile(f).metadata.num_rows:,}</td>\n'
    f'      <td class="num">{f.stat().st_size/1e6:.0f} MB</td>\n'
    f'      <td class="cols">{cols_of(f)}</td></tr>' for f in tabs)
i = h.find('id="outputs"')
j = h.find('</tbody>', i)
assert i > 0 and j > i, 'Derived tables tbody not found'
# No trailing newline: rows_html already opens each row with one, and the cleanup strips
# them with \\s*. Adding one here is not removed on the next pass and leaks a byte per run.
h = h[:j] + rows_html + h[j:]

DASH.write_text(h, encoding='utf-8')
print(f'WROTE {DASH}  ({DASH.stat().st_size/1e6:.1f} MB)  +{len(tabs)} derived-table rows')

case law: 7 tables, 50,321,215 rows, 539 MB


metric rows: 2.18 B / 28 tables -> 2.18 B / 28 tables
WROTE /project/jevans/Dawoon/Science of Science/validation/metrics_dashboard.html  (23.7 MB)  +7 derived-table rows
CPU times: user 1.87 s, sys: 168 ms, total: 2.04 s
Wall time: 1.42 s


In [5]:
%%time
# ---- 7) the nav label: "Pairs" is the paper–patent pair pipeline ---------------------------
h = DASH.read_text(encoding='utf-8')
o = '<a href="#val-ppp"><span class="dot p4"></span>Pairs</a>'
n = '<a href="#val-ppp"><span class="dot p4"></span>Patent Paper Pair</a>'
if o in h:
    h = h.replace(o, n, 1); print('nav: "Pairs" -> "Patent Paper Pair"')
elif n in h:
    print('nav already reads "Patent Paper Pair"')
else:
    raise AssertionError('the #val-ppp nav entry was not found')
DASH.write_text(h, encoding='utf-8')

# ---- final verification -------------------------------------------------------------------
h = DASH.read_text(encoding='utf-8')
checks = {
    # Two elements carry data-inject="caselaw" -- the KPI card and the raw-data row -- so the
    # attribute is counted per element, not in the aggregate.
    'case law KPI card once':      h.count('<div class="kpi" data-inject="caselaw">') == 1,
    'raw-data row once':           h.count('<tr data-inject="caselaw">') == 1,
    'derived rows once':           h.count('<tr data-pipe="caselaw">') == len(tabs),
    'nav label fixed':             h.count('>Patent Paper Pair</a>') == 1 and '>Pairs</a>' not in h,
    'five pipelines':              'Five pipelines' in h and 'Four pipelines' not in h,
    'pipebar has 5':               h.count('<div class="pipebar"><i></i><i></i><i></i><i></i><i></i></div>') == 1,
    'derived heading count':       f'<span class="meta">{21 + len(tabs)} parquet files</span>' in h,
    'section injected once':       h.count('id="val-caselaw"') == 1,
    'citation present twice':      h.count('arXiv:2410.04493') == 2,
    'p5 rules once':               h.count('.cardhead .sec.p5') == 1,
    'section tags balanced':       h.count('<section') == h.count('</section>'),
    'table tags balanced':         h.count('<table') == h.count('</table>'),
    'tr tags balanced':            h.count('<tr') == h.count('</tr>'),
}
for k, v in checks.items():
    print(f'  {"OK  " if v else "FAIL"} {k}')
assert all(checks.values()), 'dashboard update is not clean'
print(f'\nfigures {h.count("<figure")}  ·  derived-table rows {h.count("<tr data-pipe=")}  ·  '
      f'{DASH.stat().st_size/1e6:.1f} MB')

nav already reads "Patent Paper Pair"


  OK   case law KPI card once
  OK   raw-data row once
  OK   derived rows once
  OK   nav label fixed
  OK   five pipelines
  OK   pipebar has 5
  OK   derived heading count
  OK   section injected once
  OK   citation present twice
  OK   p5 rules once
  OK   section tags balanced
  OK   table tags balanced
  OK   tr tags balanced

figures 65  ·  derived-table rows 28  ·  23.7 MB
CPU times: user 257 ms, sys: 45.9 ms, total: 303 ms
Wall time: 313 ms


In [ ]:
# Refresh every gallery from current exports, including the dashboard exclusions.
import subprocess
import sys
subprocess.run([sys.executable, str(VAL / 'refresh_dashboard.py'), '--source', str(VAL), '--output', str(VAL / 'metrics_dashboard.html')], check=True)
